In [3]:
from pathlib import Path
from typing import Iterable, List, Optional, Tuple
import sys
import numpy as np


sys.path.insert(0, "../../titan_v6")
import load_functions as load_v6
from load_and_preprocessing import titan_load_and_preprocessing as titan_load_and_preprocessing_v6


def parse_experiment_name(exp_path: Path) -> Optional[str]:
    path_str = str(exp_path)
    if "KHz_U_" not in path_str:
        return None
    name = path_str.split("KHz_U_")[1]
    return name.rstrip("\\/")  # remove trailing separators


def normalize_n_a_type(n_a_type: str) -> str:
    return str(n_a_type).lower()


def resolve_ref_indices(exp_path: Path, n_a_type_norm: str, vref_ref_idx) -> Tuple[List[int], Optional[np.ndarray], Optional[dict]]:
    vref_vect = None
    dvref_tele = None
    ref_indices: List[int] = []

    if n_a_type_norm in ("v05", "v06"):
        want_all = (vref_ref_idx is None or (isinstance(vref_ref_idx, str) and vref_ref_idx.lower() == "all"))
        try:
            exp_path_vref = load_v6.find_most_recent_vref(exp_path)
            n_vrefs, vref_vect, dvref_tele = load_v6.load_vref_sweep(exp_path, return_telemetry=True)

            if dvref_tele is not None and dvref_tele.get("coarse") is not None:
                print(
                    f"Loaded dvref multi payload from {exp_path_vref.name}: "
                    f"n_vrefs={n_vrefs}, vrefs={dvref_tele.get('vrefs')}"
                )
                if dvref_tele.get("model") is not None:
                    m = dvref_tele["model"]
                    print(
                        "TTN model: "
                        f"pass_fail={m.get('pass_fail')} "
                        f"total_dcnt_x16={m.get('total_dcnt_x16')} "
                        f"threshold={m.get('coarse_dcnt_x16_threshold')}"
                    )
            else:
                print(f"Loaded vref sweep from {exp_path_vref.name}: n_vrefs={n_vrefs}")

            if want_all and n_vrefs:
                ref_indices = list(range(int(n_vrefs)))
            else:
                if isinstance(vref_ref_idx, (list, tuple, np.ndarray)):
                    ref_indices = [int(x) for x in vref_ref_idx]
                else:
                    ref_indices = [int(vref_ref_idx)]
        except Exception as exc:
            print(f"Could not load vref sweep for {exp_path.name}: {exc}")
            ref_indices = [-1 if want_all else int(vref_ref_idx)]
            vref_vect = None
            dvref_tele = None
    else:
        ref_indices = [vref_ref_idx]

    return ref_indices, vref_vect, dvref_tele


def make_slice_label(n_a_type_norm: str, ref_idx: int, vref_vect: Optional[np.ndarray]) -> Tuple[Optional[str], Optional[float], int]:
    ref_idx_norm = ref_idx
    vref_value = None

    if vref_vect is not None and len(vref_vect) > 0:
        if ref_idx_norm < 0:
            ref_idx_norm = len(vref_vect) + ref_idx_norm
        if 0 <= ref_idx_norm < len(vref_vect):
            try:
                vref_value = float(vref_vect[ref_idx_norm])
            except Exception:
                vref_value = None

    slice_label = None
    if n_a_type_norm in ("v05", "v06"):
        if vref_value is not None:
            slice_label = f"vref_idx={ref_idx_norm}_vref={vref_value:g}"
        else:
            slice_label = f"vref_idx={ref_idx_norm}"

    return slice_label, vref_value, ref_idx_norm

In [4]:
def run_experiment(exp_path: Path, n_wells: int, n_a_type: str, vref_ref_idx):
    experiment_name = parse_experiment_name(exp_path)
    if experiment_name is None:
        print(f"[SKIP] {exp_path.name}: invalid experiment directory (missing 'KHz_U_').")
        return []

    readout_files = list(exp_path.glob("*readout*.bin"))
    if not readout_files:
        find_active_files = list(exp_path.glob("*find_active*.bin"))
        if not find_active_files:
            print(f"[WARN] {exp_path.name}: no readout or find_active files found.")

    print(f"[INFO] {exp_path.name}: found {len(readout_files)} readout file(s). Processing...")

    n_a_type_norm = normalize_n_a_type(n_a_type)
    ref_indices, vref_vect, _dvref_tele = resolve_ref_indices(exp_path, n_a_type_norm, vref_ref_idx)
    is_multi_vref = len(ref_indices) > 1

    exps = []
    total_slices = len(ref_indices)

    for i_slice, ref_idx in enumerate(ref_indices, start=1):
        slice_label, _vref_value, ref_idx_norm = make_slice_label(n_a_type_norm, int(ref_idx), vref_vect)
        experiment_name_slice = experiment_name
        if slice_label is not None and (is_multi_vref or (vref_ref_idx not in (-1, "-1"))):
            experiment_name_slice = f"{experiment_name}__{slice_label}"

        print(f"[INFO] Slice {i_slice}/{total_slices}: {experiment_name_slice}")

        try:
            exps.append(
                titan_load_and_preprocessing_v6(
                    exp_path,
                    n_wells=n_wells,
                    start_type="temperature",
                    end_time_min=60,
                    n_a_type=n_a_type,
                    print_status=True,
                    plt_gain_calib=False,
                    save_gain_calib=False,
                    plot_gain_3d=False,
                    ref_idx=int(ref_idx) if n_a_type_norm in ("v05", "v06") else ref_idx,
                )
            )
            print(f"[OK] Completed slice {i_slice}/{total_slices} (ref_idx={ref_idx_norm}).")
        except Exception as exc:
            print(f"[ERROR] {exp_path.name} | ref_idx={ref_idx}: {exc}")
            print("[INFO] Skipping this ref slice and continuing...")

    return exps

In [5]:
n_wells = 10
n_a_type = "v06"
vref_ref_idx = "all"

exp_folder = Path("/vol/bitbucket/gk225/POC_DDM_multi")
exp_paths = [exp_folder / "D20260320_E00_C00_F4500KHz_U_Elena_steap_cv"]

exps_results = []
for i_path, exp_path in enumerate(exp_paths):
    print(f"RUN {i_path} -- NWELLS {n_wells} -- N_A_TYPE {n_a_type}")
    exps = run_experiment(exp_path=exp_path, n_wells=n_wells, n_a_type=n_a_type, vref_ref_idx=vref_ref_idx)
    exps_results.append(exps)



RUN 0 -- NWELLS 10 -- N_A_TYPE v06
[INFO] D20260320_E00_C00_F4500KHz_U_Elena_steap_cv: found 1 readout file(s). Processing...
Loaded vref sweep from 20260320T105348.52_vref_sweep.bin: n_vrefs=2
[INFO] Slice 1/2: Elena_steap_cv__vref_idx=0_vref=1810
EXP PATH /vol/bitbucket/gk225/POC_DDM_multi/D20260320_E00_C00_F4500KHz_U_Elena_steap_cv 
VREF PATH /vol/bitbucket/gk225/POC_DDM_multi/D20260320_E00_C00_F4500KHz_U_Elena_steap_cv/20260320T105348.52_vref_sweep.bin 
Load vref... 
Loaded vref list: n_vrefs=2, first=1810, last=2610
EXP PATH /vol/bitbucket/gk225/POC_DDM_multi/D20260320_E00_C00_F4500KHz_U_Elena_steap_cv 
READOUT PATH /vol/bitbucket/gk225/POC_DDM_multi/D20260320_E00_C00_F4500KHz_U_Elena_steap_cv/20260320T105439.73_readout_time.bin 
Load data... 
(290, 204, 2, 630)
Time vector: n=630, median_dt=4s, duration≈41.03min
(59160,)
new gain shape  (59160,)


/vol/bitbucket/gk225/POC_DDM/gk_code/outlier_detection/../../new_titan/titan/linearise.py:48: RuntimeWarning: divide by zero encountered in log
  log_div = np.log((tau_i - C) / A) / np.log((tau_f - C) / A)  # Intermediate operation
/vol/bitbucket/gk225/POC_DDM/gk_code/outlier_detection/../../new_titan/titan/linearise.py:48: RuntimeWarning: invalid value encountered in log
  log_div = np.log((tau_i - C) / A) / np.log((tau_f - C) / A)  # Intermediate operation
/vol/bitbucket/gk225/POC_DDM/gk_code/outlier_detection/../../new_titan/titan/linearise.py:48: RuntimeWarning: invalid value encountered in divide
  log_div = np.log((tau_i - C) / A) / np.log((tau_f - C) / A)  # Intermediate operation
/vol/bitbucket/gk225/POC_DDM/gk_code/outlier_detection/../../new_titan/titan/linearise.py:49: RuntimeWarning: divide by zero encountered in divide
  D = (V_i - (log_div * V_f)) / (1 - log_div)  # Calculate D
/vol/bitbucket/gk225/POC_DDM/gk_code/outlier_detection/../../new_titan/titan/linearise.py:50: R

(290, 204, 2, 630)
Data loaded. 
Preprocessing start...
Active pixel counts (ref_idx=0, vref=1810.0): lacewing=19552, non_temp=56782, gain=3637, lin=3181, combined=3027
v06 temperature trace from readout meta: n=630, finite=630, min=0, max=581
v06 temperature: no usable temp_log.bin found, plotting readout phase only (630 samples).
Preprocessing end.
[OK] Completed slice 1/2 (ref_idx=0).
[INFO] Slice 2/2: Elena_steap_cv__vref_idx=1_vref=2610
EXP PATH /vol/bitbucket/gk225/POC_DDM_multi/D20260320_E00_C00_F4500KHz_U_Elena_steap_cv 
VREF PATH /vol/bitbucket/gk225/POC_DDM_multi/D20260320_E00_C00_F4500KHz_U_Elena_steap_cv/20260320T105348.52_vref_sweep.bin 
Load vref... 
Loaded vref list: n_vrefs=2, first=1810, last=2610
EXP PATH /vol/bitbucket/gk225/POC_DDM_multi/D20260320_E00_C00_F4500KHz_U_Elena_steap_cv 
READOUT PATH /vol/bitbucket/gk225/POC_DDM_multi/D20260320_E00_C00_F4500KHz_U_Elena_steap_cv/20260320T105439.73_readout_time.bin 
Load data... 
(290, 204, 2, 630)
Time vector: n=630, media

In [6]:
dir(exps_results[0][0])

['COLS',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_wells_list',
 'exp_path_str',
 'nwells',
 'temperature_1d_v06',
 'temperature_1d_v06_with_ramp',
 'temperature_3d',
 'temperature_time_s_v06_with_ramp',
 'temperature_v06_heat_n',
 'temperature_v06_readout_n',
 'wells_list']

In [7]:
dir(exps_results[0][0].wells_list[0])

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'add_known_results_dna_TB',
 'idx_active',
 'idx_end',
 'idx_settled',
 'idx_start',
 'time',
 'time_min',
 'time_npr',
 'well_2d',
 'well_2d_active',
 'well_2d_bs',
 'well_2d_bs_active',
 'well_2d_bs_active_filt',
 'well_2d_bs_active_mean',
 'well_2d_bs_active_mean_filt',
 'well_2d_bs_filt',
 'well_2d_nl',
 'well_2d_nl_active',
 'well_2d_nl_active_mean',
 'well_2d_nl_bs',
 'well_2d_nl_bs_active',
 'well_2d_nl_bs_active_mean',
 'well_2d_nl_bs_active_mean_filt',
 'well_2d_npr',
 'well_2d_start',
 'well_2d_start_bs',
 'well_2d_start_bs_active',
 'well_2d_start_bs_active_mean',
 'well_2d_temp_npr',
 'well_3d

In [8]:
sum(exps_results[0][0].wells_list[0].idx_active)

200

In [9]:
t_0 = exps_results[0][0].wells_list[0].idx_active
idx_0 = [i for i, x in enumerate(t_0) if x]

t_1 = exps_results[0][1].wells_list[0].idx_active
idx_1 = [i for i, x in enumerate(t_1) if x]

s1 = set(idx_1)
idx_overlap = [i for i in idx_0 if i in s1]
idx_overlap, len(idx_overlap)


([], 0)

In [10]:
exps_results[0][0].wells_list[0]